In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

# Locate the ETIE project directory
PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = PROJECT_ROOT / "models" / "etie_isolation_forest_candidate.joblib"
SCHEMA_PATH = PROJECT_ROOT / "models" / "anomaly_feature_schema.json"
CALIBRATION_PATH = PROJECT_ROOT / "models" / "anomaly_calibration_scores.npy"

# Load saved artifacts
candidate_model = joblib.load(MODEL_PATH)

with open(SCHEMA_PATH, "r", encoding="utf-8") as file:
    FEATURE_COLUMNS = json.load(file)

calibration_raw_scores = np.load(CALIBRATION_PATH)

print("Saved model loaded:", type(candidate_model).__name__)
print("Feature count:", len(FEATURE_COLUMNS))
print("Calibration reference scores:", len(calibration_raw_scores))
print("Model feature count:", candidate_model.n_features_in_)

Saved model loaded: IsolationForest
Feature count: 78
Calibration reference scores: 209527
Model feature count: 78


In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "CIC-IDS-2017-V2.csv"

# Load only the 78 features and label column
required_columns = FEATURE_COLUMNS + ["label"]

data_sample = pd.read_csv(
    DATA_PATH,
    usecols=required_columns
)

print("Loaded rows:", len(data_sample))
print("Loaded columns:", len(data_sample.columns))
print("\nLabel counts:")
print(data_sample["label"].value_counts())

Loaded rows: 2695162
Loaded columns: 79

Label counts:
label
BENIGN                        2096484
DoS Hulk                       172849
Comb                           172800
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [3]:
# Separate labels without copying the full feature matrix.
benign_rows = data_sample[data_sample["label"] == "BENIGN"]
attack_rows = data_sample[data_sample["label"] != "BENIGN"]

# Reproducible sample
benign_sample = benign_rows.sample(
    n=10_000,
    random_state=42
)

attack_sample = attack_rows.sample(
    n=10_000,
    random_state=42
)

# Combine and shuffle the sampled rows
calibration_experiment_df = pd.concat(
    [benign_sample, attack_sample],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

print("Experiment sample shape:", calibration_experiment_df.shape)
print("\nLabel counts:")
print(calibration_experiment_df["label"].value_counts())

Experiment sample shape: (20000, 79)

Label counts:
label
BENIGN                        10000
Comb                           2885
DoS Hulk                       2849
DDoS                           2134
PortScan                       1565
DoS GoldenEye                   174
DoS Slowhttptest                 97
DoS slowloris                    94
FTP-Patator                      85
SSH-Patator                      59
Bot                              29
Web Attack � Brute Force         23
Web Attack � XSS                  5
Web Attack � Sql Injection        1
Name: count, dtype: int64


In [4]:
# Extract features in the model's expected order
X_experiment = calibration_experiment_df[FEATURE_COLUMNS]

# Get raw Isolation Forest scores
raw_scores = candidate_model.score_samples(X_experiment)

# Convert raw scores into anomaly scores relative to the saved benign reference
sorted_reference_scores = np.sort(calibration_raw_scores)

ranks = np.searchsorted(
    sorted_reference_scores,
    raw_scores,
    side="right"
)

anomaly_scores = 1 - ranks / len(sorted_reference_scores)

# Attach scores to the sampled records
calibration_experiment_df["anomaly_score"] = anomaly_scores
calibration_experiment_df["is_attack"] = (
    calibration_experiment_df["label"] != "BENIGN"
).astype(int)

print("Rows scored:", len(anomaly_scores))
print("All scores finite:", np.isfinite(anomaly_scores).all())
print("\nMean anomaly score by class:")
print(
    calibration_experiment_df.groupby("is_attack")["anomaly_score"]
    .mean()
    .rename(index={0: "Benign", 1: "Attack"})
)

Rows scored: 20000
All scores finite: True

Mean anomaly score by class:
is_attack
Benign    0.501390
Attack    0.854935
Name: anomaly_score, dtype: float64


In [5]:
from sklearn.metrics import roc_auc_score, brier_score_loss

y = calibration_experiment_df["is_attack"].to_numpy()
scores = calibration_experiment_df["anomaly_score"].to_numpy()

baseline_probability = np.full(len(y), y.mean())

print("Attack proportion in sample:", round(y.mean(), 4))
print("Anomaly-score ROC-AUC:", round(roc_auc_score(y, scores), 6))
print(
    "Baseline Brier score:",
    round(brier_score_loss(y, baseline_probability), 6)
)

Attack proportion in sample: 0.5
Anomaly-score ROC-AUC: 0.855299
Baseline Brier score: 0.25


In [6]:
from sklearn.model_selection import train_test_split

X_scores = calibration_experiment_df["anomaly_score"].to_numpy()
y = calibration_experiment_df["is_attack"].to_numpy()

X_cal, X_eval, y_cal, y_eval = train_test_split(
    X_scores,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

print("Calibration samples:", len(X_cal))
print("Evaluation samples:", len(X_eval))
print("Calibration attack rate:", round(y_cal.mean(), 4))
print("Evaluation attack rate:", round(y_eval.mean(), 4))

Calibration samples: 12000
Evaluation samples: 8000
Calibration attack rate: 0.5
Evaluation attack rate: 0.5


In [7]:
from sklearn.isotonic import IsotonicRegression

reliability_calibrator = IsotonicRegression(
    y_min=0.0,
    y_max=1.0,
    out_of_bounds="clip"
)

reliability_calibrator.fit(X_cal, y_cal)

print("Isotonic calibrator fitted successfully.")
print("Calibration samples used:", len(X_cal))
print("Evaluation samples kept separate:", len(X_eval))

Isotonic calibrator fitted successfully.
Calibration samples used: 12000
Evaluation samples kept separate: 8000


In [8]:
from sklearn.metrics import brier_score_loss, log_loss

# Predict attack probabilities on the held-out evaluation samples
calibrated_probabilities = reliability_calibrator.predict(X_eval)

# Constant baseline: predict the calibration subset's attack rate
baseline_probabilities = np.full(len(y_eval), y_cal.mean())

print(
    "Calibrated Brier score:",
    round(brier_score_loss(y_eval, calibrated_probabilities), 6)
)
print(
    "Baseline Brier score:",
    round(brier_score_loss(y_eval, baseline_probabilities), 6)
)
print(
    "Calibrated log loss:",
    round(log_loss(y_eval, calibrated_probabilities, labels=[0, 1]), 6)
)
print(
    "Baseline log loss:",
    round(log_loss(y_eval, baseline_probabilities, labels=[0, 1]), 6)
)
print(
    "Predicted probability range:",
    (
        round(calibrated_probabilities.min(), 4),
        round(calibrated_probabilities.max(), 4)
    )
)

Calibrated Brier score: 0.123239
Baseline Brier score: 0.25
Calibrated log loss: 0.3707
Baseline log loss: 0.693147
Predicted probability range: (np.float64(0.0), np.float64(0.9937))


In [9]:
calibration_check_df = pd.DataFrame({
    "predicted_probability": calibrated_probabilities,
    "actual_attack": y_eval
})

calibration_check_df["probability_bin"] = pd.cut(
    calibration_check_df["predicted_probability"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True
)

calibration_summary = (
    calibration_check_df
    .groupby("probability_bin", observed=True)
    .agg(
        flow_count=("actual_attack", "size"),
        mean_predicted_probability=("predicted_probability", "mean"),
        actual_attack_rate=("actual_attack", "mean")
    )
)

print(calibration_summary.round(4).to_string())

                 flow_count  mean_predicted_probability  actual_attack_rate
probability_bin                                                            
(-0.001, 0.1]          1559                      0.0000              0.0000
(0.1, 0.2]               61                      0.1368              0.1311
(0.3, 0.4]             3319                      0.3566              0.3552
(0.8, 0.9]             1898                      0.8774              0.8762
(0.9, 1.0]             1163                      0.9908              0.9888


In [10]:
CALIBRATOR_PATH = PROJECT_ROOT / "models" / "etie_isotonic_calibrator_candidate.joblib"

joblib.dump(reliability_calibrator, CALIBRATOR_PATH)

print("Candidate calibrator saved:", CALIBRATOR_PATH)
print("File exists:", CALIBRATOR_PATH.exists())
print("File size (KB):", round(CALIBRATOR_PATH.stat().st_size / 1024, 2))

Candidate calibrator saved: c:\Users\Dilum\Desktop\ETIE\models\etie_isotonic_calibrator_candidate.joblib
File exists: True
File size (KB): 0.83


In [11]:
loaded_calibrator = joblib.load(CALIBRATOR_PATH)

original_predictions = reliability_calibrator.predict(X_eval)
reloaded_predictions = loaded_calibrator.predict(X_eval)

print("Reload successful:", loaded_calibrator is not None)
print(
    "Predictions match:",
    np.allclose(original_predictions, reloaded_predictions)
)
print(
    "Maximum prediction difference:",
    float(np.max(np.abs(original_predictions - reloaded_predictions)))
)

Reload successful: True
Predictions match: True
Maximum prediction difference: 0.0


In [12]:
calibration_report = {
    "calibrator": "IsotonicRegression",
    "model": "IsolationForest",
    "model_version": "candidate_1",
    "experiment_sample_size": int(len(calibration_experiment_df)),
    "calibration_sample_size": int(len(X_cal)),
    "evaluation_sample_size": int(len(X_eval)),
    "sample_attack_rate": float(y.mean()),
    "roc_auc_anomaly_score": float(roc_auc_score(y, X_scores)),
    "calibrated_brier_score": float(
        brier_score_loss(y_eval, calibrated_probabilities)
    ),
    "baseline_brier_score": float(
        brier_score_loss(y_eval, baseline_probabilities)
    ),
    "calibrated_log_loss": float(
        log_loss(y_eval, calibrated_probabilities, labels=[0, 1])
    ),
    "baseline_log_loss": float(
        log_loss(y_eval, baseline_probabilities, labels=[0, 1])
    ),
    "reliability_status": "candidate_not_independently_validated",
    "limitations": [
        "Calibration experiment uses a balanced sample.",
        "Calibration and evaluation examples come from the same underlying dataset.",
        "Predicted probabilities are not established as real-world attack probabilities.",
        "Real-world prevalence and distribution shift have not been validated."
    ]
}

REPORT_PATH = PROJECT_ROOT / "models" / "etie_reliability_calibration_report.json"

with open(REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(calibration_report, file, indent=2)

print("Report saved:", REPORT_PATH)
print("File exists:", REPORT_PATH.exists())

Report saved: c:\Users\Dilum\Desktop\ETIE\models\etie_reliability_calibration_report.json
File exists: True
